# Data cleaning

Takes the raw workbook and turns it into one analysis-ready table for the dashboard.

Every decision here comes from `data_source_review.ipynb`. The short version of what that
notebook concluded, and what this one does about it:

| Conclusion from the review | What happens here |
|---|---|
| `Ticket Type` holds two things at once | Split into `queue` and `request_type` |
| 18 tickets do not fit that pattern | Labelled "Unclassified" so they stay visible |
| `language_dwid` defaults to English | Not used; the queue carries the language |
| Turnaround time is generated, not measured | Still calculated, but named so it is obvious |
| `processing_type` and the language table add nothing | Dropped |
| Rows are ordered by ticket type, not by date | Sorted by creation time |

No rows are added or removed. Everything is either renamed, derived or dropped.

In [ ]:
import pandas as pd

RAW = "data/raw/Casestudy-Kiwi.com-BusinessAnalyst.xlsx"
OUT = "data/clean/tickets_clean.csv"

raw = pd.read_excel(RAW, sheet_name="fact_ticket")
print("Rows in:", len(raw))
raw.head(3)

The language table is not loaded. The review showed the only field it contributes,
`language_dwid`, contradicts the queue on 17% of tickets and defaults to English, so the
language reporting is built from `Ticket Type` instead. Nothing else in that table is used.

## 1. Rename and drop

`Ticket Type` is the only column with a space and a capital letter, and `bid` is not an
obvious name. `processing_type` says "Manual" on every row, and `language_dwid` is the
field the review rejected.

In [ ]:
clean = raw.rename(columns={
    "Ticket Type": "ticket_type",
    "bid": "booking_id",
    "creation_timestamp_utc": "created_at",
    "resolution_timestamp_utc": "resolved_at",
})

clean = clean.drop(columns=["processing_type", "language_dwid"])
print(list(clean.columns))

## 2. Timestamps

They already load as real dates, but parsing them explicitly means this notebook does not
depend on Excel being read the same way everywhere.

In [ ]:
clean["created_at"] = pd.to_datetime(clean["created_at"])
clean["resolved_at"] = pd.to_datetime(clean["resolved_at"])

print(clean[["created_at", "resolved_at"]].dtypes)

## 3. Split the ticket type

`Helpdesk EN - Refund requests` is really two facts: the queue the ticket sits in, which
stands in for the language, and whether it is about a refund.

The 18 "Helpdesk communication" tickets match neither pattern. Rather than leaving them
empty, where they would silently drop out of every chart, they get their own label.

In [ ]:
clean["queue"] = clean["ticket_type"].str.extract(r"Helpdesk (?:- )?(EN|International|JA|KO)")
clean["request_type"] = clean["ticket_type"].str.extract(r"- (Refund|Non-refund) requests")

clean["queue"] = clean["queue"].fillna("Unclassified")
clean["request_type"] = clean["request_type"].fillna("Not specified")

print(clean.groupby(["queue", "request_type"]).size().to_string())

"Not specified" covers two different situations: the 18 unclassified tickets, and the
Japanese and Korean queues, which genuinely are not split by refund. Worth remembering when
reading a refund breakdown — it is not available for every queue.

## 4. Date parts for the dashboard

The dashboard groups by day, and the hour and weekday are useful for spotting when the
workload arrives. These are UTC, as the review noted.

In [ ]:
clean["created_date"] = clean["created_at"].dt.date
clean["created_hour"] = clean["created_at"].dt.hour
clean["created_weekday"] = clean["created_at"].dt.day_name()
clean["resolved_date"] = clean["resolved_at"].dt.date

clean[["created_at", "created_date", "created_hour", "created_weekday"]].head(3)

## 5. Turnaround time

Calculated the way it would be on real data. The column is called
`turnaround_minutes_generated` rather than `turnaround_minutes`, so that anyone using this
file has to notice that the review found these values to be generated rather than measured.

In [ ]:
clean["turnaround_minutes_generated"] = (
    (clean["resolved_at"] - clean["created_at"]).dt.total_seconds() / 60)

print(clean["turnaround_minutes_generated"].describe().round(1).to_string())

## 6. Refund flags

These load as `0.0` / `1.0` floats because of the 18 missing values, and converting a float
column straight to boolean would turn every missing value into `True`. Converting to a
nullable boolean keeps the gaps as gaps.

In [ ]:
for flag in ["has_accepted_refund", "has_out_of_pocket_refund"]:
    clean[flag] = clean[flag].astype("boolean")

print(clean[["has_accepted_refund", "has_out_of_pocket_refund"]].dtypes)
print("\nMissing values kept as missing:", clean["has_accepted_refund"].isna().sum())

The review also found these flags set on non-refund tickets, which suggests they describe
the booking rather than the message. They are kept for reference, but a refund count should
come from `request_type`, not from summing these.

## 7. Attributes the dashboard needs

A few things are properties of the ticket itself, so they belong here rather than being
recalculated in the dashboard: whether it arrived at the weekend, whether it was closed the
same day, and how the booking's tickets relate to each other.

`resolved_same_day` is derived from the generated resolution time, so it inherits the same
caveat as turnaround.

In [ ]:
clean["is_weekend"] = clean["created_at"].dt.dayofweek >= 5
clean["resolved_same_day"] = clean["created_date"] == clean["resolved_date"]

# Repeat contact: how many tickets this booking raised, and where this one sits in the run.
clean = clean.sort_values(["booking_id", "created_at"])
clean["tickets_on_booking"] = clean.groupby("booking_id")["ticket_id"].transform("size")
clean["is_repeat_contact_booking"] = clean["tickets_on_booking"] > 1
clean["contact_seq"] = clean.groupby("booking_id").cumcount() + 1
clean["hours_since_prev_contact"] = (
    clean.groupby("booking_id")["created_at"].diff().dt.total_seconds() / 3600)

print(clean[["tickets_on_booking", "contact_seq"]].describe().loc[["max"]].to_string())
print("\nTickets on a booking that raised more than one:",
      int(clean["is_repeat_contact_booking"].sum()))

## 8. Sort and order the columns

The raw sheet is ordered by ticket type, so anything that relies on row order would be
wrong. Sorting by creation time fixes that.

In [ ]:
COLUMNS = [
    "ticket_id", "booking_id", "ticket_type", "queue", "request_type",
    "created_at", "created_date", "created_hour", "created_weekday",
    "resolved_at", "resolved_date", "turnaround_minutes_generated",
    "has_accepted_refund", "has_out_of_pocket_refund",
    "is_weekend", "resolved_same_day",
    "tickets_on_booking", "is_repeat_contact_booking", "contact_seq",
    "hours_since_prev_contact",
]

clean = clean.sort_values("created_at").reset_index(drop=True)[COLUMNS]
clean.head(3)

## 9. Checks before saving

Nothing here should be a surprise — these confirm the cleaning did what it was meant to and
did not lose anything on the way.

Two columns are allowed to have gaps: the refund flags on the 18 unclassified tickets, and
`hours_since_prev_contact`, which has nothing to measure from on a booking's first ticket.
That second one should be empty exactly as often as there are bookings.

In [ ]:
print("Rows in / rows out:      ", len(raw), "/", len(clean))
print("ticket_id still unique:  ", clean["ticket_id"].is_unique)
print("Sorted by creation time: ", clean["created_at"].is_monotonic_increasing)
print()
missing = clean.isna().sum()
print("Missing values:")
print(missing[missing > 0].to_string() or "  none")
print("\n  the 18 refund flags are the unclassified tickets")
print("  hours_since_prev_contact is empty for the first ticket on each booking:",
      clean["hours_since_prev_contact"].isna().sum() == clean["booking_id"].nunique())
print()
print("Queues:       ", sorted(clean["queue"].unique()))
print("Request types:", sorted(clean["request_type"].unique()))

## 10. Save

In [ ]:
import os

os.makedirs("data/clean", exist_ok=True)
clean.to_csv(OUT, index=False)

print("Saved", len(clean), "rows to", OUT)
clean.head()

**What this file is ready for**

Tickets created and resolved per day, broken down by queue and request type, plus the
hour-of-day and weekday views and the repeat-contact rate from `booking_id`.

**What it deliberately does not support**

Backlog and net ticket change, because every ticket in the extract is already resolved, and
any reading of turnaround time as real performance.

## 11. Rebuild the aggregates

`tickets_clean.csv` is the only file that has to be maintained, but the dashboard also
reads a set of aggregate tables under `data/clean/derived/`. They are derived entirely
from the file just written, so re-running this notebook without rebuilding them would
leave the dashboard showing figures from the previous run.

Running the build step here keeps the chain in one place: clean the data, then rebuild
everything that depends on it.

In [ ]:
import subprocess
import sys

result = subprocess.run([sys.executable, "pipeline/build.py"],
                        capture_output=True, text=True)
print(result.stdout[-1500:] if result.stdout else "")
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise SystemExit("build.py failed - the derived tables are now out of date")
print("\nDerived tables rebuilt from the clean file just saved.")